<a href="https://colab.research.google.com/github/AjinkyaD3/test-cd/blob/main/dbatu_extractor_colab_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DBATU Academic Result PDF Extractor

This notebook extracts student records from DBATU examination result PDFs and exports them into a clean, structured Excel file.

## How to use this in Jupyter Notebook:
1. Run **Cell 1** to install the required libraries.
2. Run **Cell 2** to load the extraction logic.
3. Run **Cell 3**. A **"Choose Files"** button will appear below the cell.
4. Click it to **upload your PDF file(s)** from your computer.
5. The script will process the PDFs. Once finished, the structured **Excel file will automatically download** to your computer!

In [1]:
# Cell 1: Install required dependencies
!pip install -q pdfplumber pandas openpyxl ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 50.4 MB/s eta 0:00:00


In [2]:
# Cell 2: Load the Extractor Logic
import json
import os
import re
from pathlib import Path
import pandas as pd
import pdfplumber
from google.colab import files

# =============================================================================
# Parsing Functions
# =============================================================================

def extract_metadata(page1_lines: list[str]) -> dict:
    metadata = {"university": "", "program": "", "semester": 0, "session": "", "year": 0, "exam_type": ""}
    if page1_lines:
        metadata["university"] = page1_lines[0].strip()
    for line in page1_lines[:5]:
        m = re.search(r"EXAMINATION\s*:\s*(.+?)\s+SEMESTER\s*-\s*(\d+)\s+(\w+)\s+(\d{4})\s*\(\s*(\w+)\s*\)\s*Institute\s*Record", line)
        if m:
            metadata["program"] = m.group(1).strip()
            metadata["semester"] = int(m.group(2))
            metadata["session"] = m.group(3).strip()
            metadata["year"] = int(m.group(4))
            metadata["exam_type"] = m.group(5).strip()
            break
    return metadata

def discover_subjects(page1_lines: list[str]) -> list[dict]:
    subjects = []
    start_idx, end_idx = None, None
    for i, line in enumerate(page1_lines):
        if "EXAMINATION" in line:
            start_idx = i + 1
        if start_idx is not None and "Seat No." in line:
            end_idx = i
            break
    if start_idx is None or end_idx is None:
        return subjects
    subject_text = " ".join(page1_lines[start_idx:end_idx])
    code_pattern = re.compile(r"([A-Z0-9]{6,})\s*:\s*")
    code_matches = list(code_pattern.finditer(subject_text))
    for idx, cm in enumerate(code_matches):
        code = cm.group(1)
        name_start = cm.end()
        name_end = code_matches[idx + 1].start() if idx + 1 < len(code_matches) else len(subject_text)
        name_text = subject_text[name_start:name_end].strip()
        credit_match = re.search(r"\(Credit\s*:\s*(\d+)\s*\)\s*$", name_text)
        if credit_match:
            credit = int(credit_match.group(1))
            name = name_text[:credit_match.start()].strip()
        else:
            credit = 0
            name = re.sub(r"\s*\)\s*$", "", name_text.strip())
        subjects.append({"code": code, "name": name, "credit": credit})
    return subjects

def parse_marking_scheme(header_lines: list[str], subjects: list[dict]) -> dict:
    scheme = {"subject_order": [], "credits": [], "ese_max": [], "ca_max": [], "mid_max": [], "total_max": [], "grade_only_indices": set(), "total_marks_max": 0, "total_grade_points_max": 0.0, "total_credits_max": 0.0}
    for line in header_lines:
        m = re.search(r"Tot\.GrP\.\-?([\d.]+)\s+Cr\.([\d.]+)", line)
        if m:
            scheme["total_grade_points_max"] = float(m.group(1))
            scheme["total_credits_max"] = float(m.group(2))
        m = re.search(r"Total\s*Marks\((\d+)\)", line)
        if m:
            scheme["total_marks_max"] = int(m.group(1))
        if line.strip().startswith("CREDIT"):
            scheme["credits"] = [int(c) for c in re.findall(r"\b(\d+)\b", line.split("CREDIT", 1)[1])]
        if re.match(r"\s*(EXT|ESE)\s+TOTAL", line):
            scheme["ese_max"] = [(int(p[0]), int(p[1])) for p in [f.split("/") for f in re.findall(r"(\d+/\d+)", line)]]
        if "CA INTERNAL" in line:
            scheme["ca_max"] = [int(n) for n in re.findall(r"\b(\d+)\b", line.split("CA INTERNAL", 1)[1])]
        if "MID INTERNAL" in line:
            scheme["mid_max"] = [int(n) for n in re.findall(r"\b(\d+)\b", line.split("MID INTERNAL", 1)[1])]
        if re.match(r"\s*TOTAL\s+", line) and "TOTAL" in line and "Marks" not in line:
            tokens = line.split("TOTAL", 1)[1].strip().split()
            total_entries, grade_indices, idx = [], set(), 0
            for token in tokens:
                if "/" in token:
                    parts = token.split("/")
                    total_entries.append((int(parts[0]), int(parts[1])))
                    idx += 1
                elif token == "GRADE":
                    total_entries.append(("GRADE", "GRADE"))
                    grade_indices.add(idx)
                    idx += 1
            scheme["total_max"] = total_entries
            scheme["grade_only_indices"] = grade_indices
    return scheme

HEADER_MARKERS = ["Dr. Babasaheb Ambedkar Technological University", "EXAMINATION :", "Seat No.", "SGPA", "Center Code", "CORE Tot.GrP.", "CREDIT", "EXT TOTAL", "ESE TOTAL", "CA INTERNAL", "MID INTERNAL", "INT TOTAL", "Total Marks("]
FOOTER_MARKERS = ["Cancel Seat No", "GRADE:", "Note :-", "AOO =", "Print By"]

def is_header_line(line: str) -> bool:
    stripped = line.strip()
    if not stripped: return False
    if any(m in stripped for m in HEADER_MARKERS): return True
    if re.match(r"^\s*TOTAL\s+(\d+/\d+|GRADE)", stripped): return True
    return False

def is_footer_line(line: str) -> bool:
    stripped = line.strip()
    return any(m in stripped for m in FOOTER_MARKERS)

def extract_data_lines(page_text: str) -> list[str]:
    lines, data_lines, in_data = page_text.split("\n"), [], False
    for line in lines:
        stripped = line.strip()
        if not stripped: continue
        if is_footer_line(stripped): break
        if not in_data:
            if re.match(r"^\s*TOTAL\s+(\d+/\d+|GRADE)", stripped):
                in_data = True
                continue
            if is_header_line(stripped): continue
            if re.match(r"^[\dA-Z\s]+$", stripped) and len(stripped) < 40: continue
            if re.match(r"^[A-Z]?\d{13,16}\s+", stripped):
                in_data = True
                data_lines.append(stripped)
        else:
            data_lines.append(stripped)
    return data_lines

SEAT_PATTERN = re.compile(r"^([A-Z]?\d{13,16})\s+(.+?)\s+(\d{5})\s*-\s*(.+?)\s+(PASS|FAIL|ATKT|WITHHELD)\s*$")

def is_student_start(line: str) -> bool:
    return bool(SEAT_PATTERN.match(line.strip()))

def parse_student_info(line: str) -> dict | None:
    m = SEAT_PATTERN.match(line.strip())
    if not m: return None
    gender, name = ("F", m.group(2).strip()[3:].strip()) if re.match(r"^\(F\)\s*(.+)$", m.group(2).strip()) else ("", m.group(2).strip())
    return {"seat_no": m.group(1), "name": name, "gender": gender, "institute_code": m.group(3), "institute_name": m.group(4).strip(), "result": m.group(5)}

def safe_int(val: str) -> int | None:
    return 0 if val == "ZOO" else (None if val == "AB" else int(val) if val.isdigit() else None)

def parse_student_block(block: list[str], subjects: list[dict], scheme: dict, metadata: dict) -> dict | None:
    if len(block) < 6: return None
    info = parse_student_info(block[0])
    if not info: return None
    record = dict(info)

    # L1: ESE + SGPA
    stripped = block[1].strip()
    sgpa_match = re.search(r"\s+(\d+\.\d{2})\s*$", stripped)
    record["sgpa"] = float(sgpa_match.group(1)) if sgpa_match else None
    if sgpa_match: stripped = stripped[:sgpa_match.start()].strip()
    ese_values = re.findall(r"\b(\d{2,3}|ZOO|AB)\b", stripped)

    # L2: CA
    m = re.match(r"^(\d{5})\s+\((Whole|Part)\)\s+(.+)$", block[2].strip())
    if m:
        record["center_code"], record["center_type"] = m.group(1), m.group(2)
        rest = m.group(3).strip()
        gp_cr_match = re.search(r"\s+([\d.]+)\s+([\d.]+)\s*$", rest)
        record["total_grade_points"] = float(gp_cr_match.group(1)) if gp_cr_match else None
        record["total_credits"] = float(gp_cr_match.group(2)) if gp_cr_match else None
        if gp_cr_match: rest = rest[:gp_cr_match.start()].strip()
        ca_values = re.findall(r"\b(\d{2,3}|ZOO|AB)\b", rest)
    else:
        record.update({"center_code": "", "center_type": "", "total_grade_points": None, "total_credits": None})
        ca_values = []

    # L3: Total marks
    record["total_marks"] = int(block[3].strip()) if re.match(r"^\d{2,4}$", block[3].strip()) else None

    grade_line_idx = next((i for i in range(len(block)-1, 4, -1) if re.search(r"\d+\.?\d*/[A-Z]{2}/[\d.]+|AU", block[i])), None)
    totals_line_idx = grade_line_idx - 1 if grade_line_idx else (len(block) - 2 if len(block) >= 7 else None)
    grade_line_idx = grade_line_idx or (len(block) - 1 if len(block) >= 7 else None)

    mid_values = re.findall(r"\b(\d{2,3}|ZOO|AB)\b", block[4].strip()) if 4 < len(block) else []
    subject_totals = []
    if totals_line_idx and totals_line_idx < len(block):
        cleaned = re.sub(r"\|", "GRADE_ONLY", block[totals_line_idx].strip())
        subject_totals = [p if p == "GRADE_ONLY" else p for p in cleaned.split() if p == "GRADE_ONLY" or re.match(r"^\d{2,3}$", p)]

    grades = []
    if grade_line_idx and grade_line_idx < len(block):
        tokens = block[grade_line_idx].strip().split()
        i = 0
        while i < len(tokens):
            if tokens[i] == "AU":
                grades.append({"grade_value": None, "grade_letter": "AU", "grade_points": None, "grace_marks": None})
                i += 1
                continue
            gm = re.match(r"(\d+\.?\d*)/([A-Z]{2})/([\d.]+)(?:\(G-(\d+)\))?", tokens[i])
            if gm:
                grades.append({"grade_value": float(gm.group(1)), "grade_letter": gm.group(2), "grade_points": float(gm.group(3)), "grace_marks": int(gm.group(4)) if gm.group(4) else None})
            i += 1

    grade_only_indices = scheme.get("grade_only_indices", set())
    ese_idx = ca_idx = mid_idx = total_idx = grade_idx = 0

    subjects_with_mid = []
    for i, subj in enumerate(subjects):
        if subj.get("grade_only", False): continue
        ese_m = scheme["ese_max"][i][0] if i < len(scheme.get("ese_max", [])) else 0
        ca_m = scheme["ca_max"][i] if i < len(scheme.get("ca_max", [])) else 0
        total_t = scheme["total_max"][i] if i < len(scheme.get("total_max", [])) else 0
        total_m = total_t[0] if isinstance(total_t, tuple) and total_t != ("GRADE", "GRADE") else 0
        if (total_m - ese_m - ca_m) > 0: subjects_with_mid.append(i)

    for i, subj in enumerate(subjects):
        code = subj["code"]
        if i in grade_only_indices or subj.get("grade_only", False):
            record.update({f"{code}_ese": None, f"{code}_ca": None, f"{code}_mid": None, f"{code}_total": None, f"{code}_grade_letter": None, f"{code}_grade_value": None, f"{code}_grade_points": None, f"{code}_grace_marks": None})
            if grade_idx < len(grades):
                g = grades[grade_idx]
                record.update({f"{code}_grade_letter": g["grade_letter"], f"{code}_grade_value": g["grade_value"], f"{code}_grade_points": g["grade_points"], f"{code}_grace_marks": g["grace_marks"]})
                grade_idx += 1
            if total_idx < len(subject_totals) and subject_totals[total_idx] == "GRADE_ONLY":
                total_idx += 1
            continue

        record[f"{code}_ese"] = safe_int(ese_values[ese_idx]) if ese_idx < len(ese_values) else None
        if record[f"{code}_ese"] is not None: ese_idx += 1
        else: ese_idx += 1

        record[f"{code}_ca"] = safe_int(ca_values[ca_idx]) if ca_idx < len(ca_values) else None
        if record[f"{code}_ca"] is not None or ca_idx < len(ca_values): ca_idx += 1

        if i in subjects_with_mid:
            record[f"{code}_mid"] = safe_int(mid_values[mid_idx]) if mid_idx < len(mid_values) else None
            mid_idx += 1
        else:
            record[f"{code}_mid"] = None

        if total_idx < len(subject_totals):
            val = subject_totals[total_idx]
            record[f"{code}_total"] = None if val == "GRADE_ONLY" else safe_int(val)
            total_idx += 1
        else:
            record[f"{code}_total"] = None

        if grade_idx < len(grades):
            g = grades[grade_idx]
            record.update({f"{code}_grade_letter": g["grade_letter"], f"{code}_grade_value": g["grade_value"], f"{code}_grade_points": g["grade_points"], f"{code}_grace_marks": g["grace_marks"]})
            grade_idx += 1
        else:
            record.update({f"{code}_grade_letter": None, f"{code}_grade_value": None, f"{code}_grade_points": None, f"{code}_grace_marks": None})

    return record

def process_pdf(pdf_path: str) -> dict:
    print(f"\nProcessing: {pdf_path}")
    with pdfplumber.open(pdf_path) as pdf:
        page1_text = pdf.pages[0].extract_text()
        page1_lines = page1_text.split("\n")
        metadata = extract_metadata(page1_lines)
        subjects = discover_subjects(page1_lines)
        header_lines = [l.strip() for l in page1_lines if is_header_line(l.strip()) or re.match(r"^\s*TOTAL\s+(\d+/\d+|GRADE)", l.strip())]
        scheme = parse_marking_scheme(header_lines, subjects)
        for idx in scheme["grade_only_indices"]:
            if idx < len(subjects): subjects[idx]["grade_only"] = True
        for s in subjects:
            if "grade_only" not in s: s["grade_only"] = s.get("credit", 0) == 0

        all_students = []
        for page in pdf.pages:
            data_lines = extract_data_lines(page.extract_text())
            blocks, curr = [], []
            for line in data_lines:
                if is_student_start(line):
                    if curr: blocks.append(curr)
                    curr = [line]
                elif curr:
                    curr.append(line)
            if curr: blocks.append(curr)
            for block in blocks:
                student = parse_student_block(block, subjects, scheme, metadata)
                if student: all_students.append(student)

    return {"metadata": metadata, "subjects": subjects, "students": all_students}

def build_summary(students: list[dict], subjects: list[dict]) -> dict:
    total = len(students)
    passed = sum(1 for s in students if s["result"] == "PASS")
    sgpa_values = [s["sgpa"] for s in students if s["sgpa"] is not None]
    avg_sgpa = sum(sgpa_values) / len(sgpa_values) if sgpa_values else 0
    return {"total_students": total, "passed": passed, "pass_percentage": round(passed/total*100, 2) if total else 0, "avg_sgpa": round(avg_sgpa, 2)}

def reshape_long(students: list[dict], subjects: list[dict]) -> pd.DataFrame:
    """Turn the wide per-subject columns (code_ese, code_ca, ...) into a
    tall/long table: one row per (student, subject) instead of one row per
    student with dozens of subject-specific columns."""
    info_cols = ["seat_no", "name", "gender", "institute_code", "institute_name", "center_code", "center_type", "result", "sgpa", "total_grade_points", "total_credits", "total_marks"]
    field_suffixes = ["ese", "ca", "mid", "total", "grade_letter", "grade_value", "grade_points", "grace_marks"]
    subj_lookup = {s["code"]: s["name"] for s in subjects}

    rows = []
    for student in students:
        base = {c: student.get(c) for c in info_cols}
        for code in subj_lookup:
            # Skip subjects this student has no columns for at all
            if not any(f"{code}_{suf}" in student for suf in field_suffixes):
                continue
            row = dict(base)
            row["subject_code"] = code
            row["subject_name"] = subj_lookup[code]
            for suf in field_suffixes:
                row[suf] = student.get(f"{code}_{suf}")
            rows.append(row)

    cols = info_cols + ["subject_code", "subject_name"] + field_suffixes
    return pd.DataFrame(rows, columns=cols)

def export_xlsx(data: dict, output_path: str):
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        pd.DataFrame([data["metadata"]]).to_excel(writer, sheet_name="Metadata", index=False)
        pd.DataFrame(data["subjects"]).to_excel(writer, sheet_name="Subjects", index=False)
        if data["students"]:
            # Long/tall format: one row per student per subject, not 8 new
            # columns per subject. This is the sheet you'll actually use.
            long_df = reshape_long(data["students"], data["subjects"])
            long_df.to_excel(writer, sheet_name="Student Results", index=False)
        pd.DataFrame([build_summary(data["students"], data["subjects"])]).to_excel(writer, sheet_name="Summary", index=False)


In [4]:
# Cell 3: Upload and Process
print("Please upload your DBATU PDF Result files:")
uploaded = files.upload()

for file_name in uploaded.keys():
    print(f"\nProcessing {file_name}...")
    try:
        # Extract data
        data = process_pdf(file_name)

        # Create base name for output file
        prog = data["metadata"].get("program", "Result")
        sem = data["metadata"].get("semester", "X")
        short_prog = "AI_DS" if "Artificial Intelligence" in prog else prog[:10].replace(" ", "_")
        out_name = f"{short_prog}_Sem{sem}_Extracted.xlsx"

        # Export and Download
        export_xlsx(data, out_name)
        print(f"\n✅ Successfully created {out_name}!")
        print(f"Extracted {len(data['students'])} students.")
        print("Downloading file now...")
        files.download(out_name)

    except Exception as e:
        print(f"\n❌ Error processing {file_name}: {e}")


Please upload your DBATU PDF Result files:


Saving Bachelor of Technology (Artificial Intelligence and Data Science)_3(DECEMBER_2025) - CR Report - Copy (1).pdf to Bachelor of Technology (Artificial Intelligence and Data Science)_3(DECEMBER_2025) - CR Report - Copy (1).pdf

Processing Bachelor of Technology (Artificial Intelligence and Data Science)_3(DECEMBER_2025) - CR Report - Copy (1).pdf...

Processing: Bachelor of Technology (Artificial Intelligence and Data Science)_3(DECEMBER_2025) - CR Report - Copy (1).pdf

✅ Successfully created AI_DS_Sem3_Extracted.xlsx!
Extracted 225 students.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>